In [2]:
import pandas as pd
import numpy as np
import time

In [33]:
N_STATES = 6   # the length of the 1 dimensional world
ACTIONS = ['left', 'right']     # available actions
EPSILON = 0.9   # greedy police
ALPHA = 0.1     # learning rate
GAMMA = 0.9    # discount factor
MAX_EPISODES = 13   # maximum episodes
FRESH_TIME = 0.3    # fresh time for one move

In [25]:
def build_q_table(n_states, actions):
    table = pd.DataFrame(
        np.zeros((n_states, len(actions))),     # q_table initial values
        columns=actions,    # actions's name
    )
    # print(table)    # show table
    return table

In [28]:
a = build_q_table(6,[1,2])

In [29]:
a

,1,2
0,0.0,0.0
1,0.0,0.0
2,0.0,0.0
3,0.0,0.0
4,0.0,0.0
5,0.0,0.0


In [72]:
def choose_action(state, q_table):
    """Choose action and return integer index (0, 1, 2, ...)"""
    state_actions = q_table.iloc[state, :]
    
    if np.random.uniform() > EPSILON or state_actions.sum() == 0:
        # Random action - return random index
        action_index = np.random.choice(len(state_actions))  # Returns 0 or 1
    else:
        # Greedy action - get position of maximum value
        action_index = state_actions.values.argmax()  # Use .values.argmax() for position
        # OR: action_index = np.argmax(state_actions.values)
    
    return int(action_index)  # Return 0 or 1, not 'left' or 'right'

In [73]:
def get_env_feedback(S,A):


    if A == 'right':
        if S == N_STATES - 2:
            R = 1
            S_ = 'terminal'
        else:
            R = 0
            if S == N_STATES:
                S_ = S
            else:
                S_ = S + 1

    else:

        R = 0
        if S == 0:
            S_ = 0
        else:
            S_ = S - 1

    return S_, R

In [74]:
def update_env(S, episode, step_counter):
    # This is how environment be updated
    env_list = ['-']*(N_STATES-1) + ['T']   # '---------T' our environment
    if S == 'terminal':
        interaction = 'Episode %s: total_steps = %s' % (episode+1, step_counter)
        print('\r{}'.format(interaction), end='')
        time.sleep(2)
        print('\r                                ', end='')
    else:
        env_list[S] = 'o'
        interaction = ''.join(env_list)
        print('\r{}'.format(interaction), end='')
        time.sleep(FRESH_TIME)

In [77]:
def rl():
    q_table = build_q_table(N_STATES, ACTIONS)
    
    for episode in range(8):
        step_counter = 0
        S = 0
        is_terminated = False
        update_env(S, episode, step_counter)
        
        while not is_terminated:
            A_index = choose_action(S, q_table)  # Returns 0 or 1
            A_name = ACTIONS[A_index]  # Convert to 'left' or 'right'
            
            S_, R = get_env_feedback(S, A_name)  # Pass action name to environment
            
            # Use action name for Q-table access
            q_predict = q_table.loc[S, A_name]  # Use 'left' or 'right'
            
            if S_ != 'terminal':
                q_target = R + GAMMA * q_table.iloc[S_, :].max()
            else:
                q_target = R
                is_terminated = True
                
            # Update using action name
            q_table.loc[S, A_name] += ALPHA * (q_target - q_predict)
            S = S_
            update_env(S, episode, step_counter+1)
            step_counter += 1
    
    return q_table

In [78]:
if __name__ == "__main__":
    q_table = rl()
    print('\r\nQ-table:\n')
    print(q_table)

                                
Q-table:

   left     right
0   0.0  0.000283
1   0.0  0.003663
2   0.0  0.030854
3   0.0  0.168206
4   0.0  0.569533
5   0.0  0.000000
